In [2]:
import pandas as pd

train_path = "/kaggle/input/competitions/shopee-product-matching/train.csv"
train = pd.read_csv(train_path)

print(train.shape)

(34250, 5)


In [9]:
print("train:", "train" in globals())
print("sample_images:", "sample_images" in globals())
print("sample_embeddings:", "sample_embeddings" in globals())
print("sample_embeddings18:", "sample_embeddings18" in globals())
print("image_pairs:", "image_pairs" in globals())

train: True
sample_images: True
sample_embeddings: False
sample_embeddings18: False
image_pairs: False


In [10]:
import torch
import numpy as np
from PIL import Image
from torchvision import models
from sklearn.preprocessing import normalize
import os

weights = models.ResNet50_Weights.DEFAULT

resnet = models.resnet50(weights=weights)
resnet.fc = torch.nn.Identity()
resnet.eval()

preprocess = weights.transforms()

image_dir = "/kaggle/input/competitions/shopee-product-matching/train_images"

sample_embeddings = []

for i, filename in enumerate(sample_images):

    image_path = os.path.join(image_dir, filename)

    image = Image.open(image_path).convert("RGB")
    image_tensor = preprocess(image).unsqueeze(0)

    with torch.no_grad():
        embedding = resnet(image_tensor)

    sample_embeddings.append(embedding.squeeze(0).numpy())

    if (i + 1) % 500 == 0:
        print(f"Processed {i + 1}/5000")

sample_embeddings = np.array(sample_embeddings)

print("Embedding shape:", sample_embeddings.shape)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 193MB/s]


Processed 500/5000
Processed 1000/5000
Processed 1500/5000
Processed 2000/5000
Processed 2500/5000
Processed 3000/5000
Processed 3500/5000
Processed 4000/5000
Processed 4500/5000
Processed 5000/5000
Embedding shape: (5000, 2048)


In [11]:
image_to_index = {
    filename: i
    for i, filename in enumerate(sample_images)
}

sample_meta = (
    train[train["image"].isin(sample_images)]
    [["image", "title", "label_group"]]
    .drop_duplicates("image")
    .copy()
)

sample_meta["embedding_index"] = sample_meta["image"].map(image_to_index)

sample_meta = sample_meta.dropna(subset=["embedding_index"])

sample_meta["embedding_index"] = (
    sample_meta["embedding_index"].astype(int)
)

print("Images with metadata:", len(sample_meta))
print()
print(sample_meta.head())


Images with metadata: 5000

                                   image  \
6   00144a49c56599d45354a1c28104c039.jpg   
8   0019a3c6755a194cb2e2c12bfc63972e.jpg   
32  004e512a55c80c07ae0684c58f5fcfd9.jpg   
35  0052b0b2e8233836f2875a14a9808b92.jpg   
36  005742b269e0ea4b352235b084e88eb4.jpg   

                                                title  label_group  \
6                            Jubah anak size 1-12 thn   1835033137   
8   [LOGU] Tempelan kulkas magnet angka, tempelan ...   2359912463   
32  Alat Facial Wajah 5 in 1 / Face Beauty Care Ma...   4266903097   
35                         CHIL KID GOLD VANILA 800GR   1632207687   
36  Laneige Cica Sleeping Mask Sample 10ml Origina...   2653016647   

    embedding_index  
6              2744  
8              4420  
32             4756  
35             3576  
36             2107  


In [12]:
import numpy as np

np.random.seed(42)

groups = sample_meta.groupby("label_group")["embedding_index"].apply(list)

positive_pairs = []

for indices in groups:
    if len(indices) >= 2:
        for i in range(len(indices)):
            for j in range(i + 1, len(indices)):
                positive_pairs.append((indices[i], indices[j]))

print("Positive pairs:", len(positive_pairs))

# Limit positives if necessary
positive_pairs = np.array(positive_pairs)

# Create negative pairs
all_indices = sample_meta["embedding_index"].values

negative_pairs = []

while len(negative_pairs) < len(positive_pairs):

    i, j = np.random.choice(all_indices, 2, replace=False)

    group_i = sample_meta.loc[
        sample_meta["embedding_index"] == i, "label_group"
    ].iloc[0]

    group_j = sample_meta.loc[
        sample_meta["embedding_index"] == j, "label_group"
    ].iloc[0]

    if group_i != group_j:
        negative_pairs.append((i, j))

negative_pairs = np.array(negative_pairs)

# Combine
image_pairs = np.vstack([
    positive_pairs,
    negative_pairs
])

image_pair_labels = np.concatenate([
    np.ones(len(positive_pairs)),
    np.zeros(len(negative_pairs))
])

print("Total pairs:", len(image_pairs))
print("Positive:", sum(image_pair_labels == 1))
print("Negative:", sum(image_pair_labels == 0))

Positive pairs: 1603
Total pairs: 3206
Positive: 1603
Negative: 1603


In [13]:
from sklearn.metrics.pairwise import cosine_similarity

image_similarities = np.array([
    cosine_similarity(
        sample_embeddings[i].reshape(1, -1),
        sample_embeddings[j].reshape(1, -1)
    )[0, 0]
    for i, j in image_pairs
])

print("Image similarities:", image_similarities.shape)
print("Minimum:", image_similarities.min())
print("Maximum:", image_similarities.max())
print("Mean:", image_similarities.mean())

Image similarities: (3206,)
Minimum: 0.043415077
Maximum: 0.9999645
Mean: 0.4574003


In [14]:
print("Character vectorizer:", "vectorizer_char" in globals())
print("Character TF-IDF vectors:", "title_vectors_char" in globals())

Character vectorizer: False
Character TF-IDF vectors: False


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer_char = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=3,
    max_features=50000
)

title_vectors_char = vectorizer_char.fit_transform(train["title"])

print("TF-IDF shape:", title_vectors_char.shape)

TF-IDF shape: (34250, 50000)


In [16]:
# Map each image filename to its TF-IDF vector
image_to_train_index = (
    train.reset_index()
    .drop_duplicates("image")
    .set_index("image")["index"]
    .to_dict()
)

sample_text_indices = [
    image_to_train_index[image]
    for image in sample_images
]

sample_title_vectors = title_vectors_char[sample_text_indices]

print("Sample title vectors:", sample_title_vectors.shape)

Sample title vectors: (5000, 50000)


In [17]:
text_similarities = np.array([
    cosine_similarity(
        sample_title_vectors[i].reshape(1, -1),
        sample_title_vectors[j].reshape(1, -1)
    )[0, 0]
    for i, j in image_pairs
])

print("Text similarities:", text_similarities.shape)
print("Minimum:", text_similarities.min())
print("Maximum:", text_similarities.max())
print("Mean:", text_similarities.mean())

Text similarities: (3206,)
Minimum: 0.0
Maximum: 1.0000000000000004
Mean: 0.25450911796854836


In [18]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

multimodal_scores = (
    0.5 * text_similarities +
    0.5 * image_similarities
)

thresholds = np.arange(0.05, 0.96, 0.05)

best_threshold = 0
best_f1 = 0
best_metrics = None

for threshold in thresholds:

    predictions = (multimodal_scores >= threshold).astype(int)

    precision = precision_score(image_pair_labels, predictions)
    recall = recall_score(image_pair_labels, predictions)
    f1 = f1_score(image_pair_labels, predictions)
    accuracy = accuracy_score(image_pair_labels, predictions)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold
        best_metrics = (accuracy, precision, recall, f1)

print("Best threshold:", best_threshold)
print("Accuracy:", best_metrics[0])
print("Precision:", best_metrics[1])
print("Recall:", best_metrics[2])
print("F1:", best_metrics[3])

Best threshold: 0.3
Accuracy: 0.9797255146600125
Precision: 0.9867088607594937
Recall: 0.9725514660012476
F1: 0.979579013509268


In [19]:
weights = [0.0, 0.25, 0.50, 0.75, 1.0]

results = []

for text_weight in weights:

    image_weight = 1 - text_weight

    scores = (
        text_weight * text_similarities +
        image_weight * image_similarities
    )

    best_f1 = 0
    best_threshold = 0
    best_precision = 0
    best_recall = 0
    best_accuracy = 0

    for threshold in np.arange(0.05, 0.96, 0.05):

        predictions = (scores >= threshold).astype(int)

        precision = precision_score(image_pair_labels, predictions)
        recall = recall_score(image_pair_labels, predictions)
        f1 = f1_score(image_pair_labels, predictions)
        accuracy = accuracy_score(image_pair_labels, predictions)

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
            best_precision = precision
            best_recall = recall
            best_accuracy = accuracy

    results.append([
        text_weight,
        image_weight,
        best_threshold,
        best_accuracy,
        best_precision,
        best_recall,
        best_f1
    ])

results

[[0.0,
  1.0,
  np.float64(0.4),
  0.9204616344354336,
  0.9170792079207921,
  0.924516531503431,
  0.9207828518173345],
 [0.25,
  0.75,
  np.float64(0.35000000000000003),
  0.9585152838427947,
  0.9542645241038319,
  0.9631940112289458,
  0.9587084756286868],
 [0.5,
  0.5,
  np.float64(0.3),
  0.9797255146600125,
  0.9867088607594937,
  0.9725514660012476,
  0.979579013509268],
 [0.75,
  0.25,
  np.float64(0.2),
  0.9862757330006239,
  0.9949206349206349,
  0.9775421085464754,
  0.9861548143486469],
 [1.0,
  0.0,
  np.float64(0.1),
  0.9778540237055521,
  0.9916559691912709,
  0.9638178415470992,
  0.9775387535590003]]

In [20]:
from sklearn.model_selection import train_test_split

X = np.column_stack([
    text_similarities,
    image_similarities
])

y = image_pair_labels

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training pairs:", len(X_train))
print("Test pairs:", len(X_test))

Training pairs: 2564
Test pairs: 642


In [21]:
from sklearn.linear_model import LogisticRegression

fusion_model = LogisticRegression(random_state=42)

fusion_model.fit(X_train, y_train)

print("Model coefficients:")
print("Text similarity:", fusion_model.coef_[0][0])
print("Image similarity:", fusion_model.coef_[0][1])
print("Intercept:", fusion_model.intercept_[0])

Model coefficients:
Text similarity: 11.06255175568112
Image similarity: 7.703202123087519
Intercept: -5.148145877200481


In [22]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

test_probabilities = fusion_model.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.05, 0.96, 0.05)

best_threshold = 0
best_f1 = 0
best_metrics = None

for threshold in thresholds:

    predictions = (test_probabilities >= threshold).astype(int)

    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions)
    recall = recall_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold
        best_metrics = (accuracy, precision, recall, f1)

print("Best threshold:", best_threshold)
print("Accuracy:", best_metrics[0])
print("Precision:", best_metrics[1])
print("Recall:", best_metrics[2])
print("F1:", best_metrics[3])

Best threshold: 0.3
Accuracy: 0.985981308411215
Precision: 0.9785276073619632
Recall: 0.9937694704049844
F1: 0.9860896445131375


In [23]:
ablation_results = [
    ["Text only", 1.0, 0.0, 0.9775387535590003],
    ["Image only", 0.0, 1.0, 0.9207828518173345],
    ["Multimodal (75/25)", 0.75, 0.25, 0.9861548143486469],
    ["Learned fusion", "Learned", "Learned", 0.9860896445131375]
]

print("ABLATION STUDY")
print("=" * 70)

for row in ablation_results:
    print(
        f"{row[0]:25} | "
        f"Text: {row[1]:>7} | "
        f"Image: {row[2]:>7} | "
        f"F1: {row[3]:.4f}"
    )

ABLATION STUDY
Text only                 | Text:     1.0 | Image:     0.0 | F1: 0.9775
Image only                | Text:     0.0 | Image:     1.0 | F1: 0.9208
Multimodal (75/25)        | Text:    0.75 | Image:    0.25 | F1: 0.9862
Learned fusion            | Text: Learned | Image: Learned | F1: 0.9861


In [24]:
# Best multimodal configuration
final_scores = (
    0.75 * text_similarities +
    0.25 * image_similarities
)

text_predictions = (text_similarities >= 0.10).astype(int)
image_predictions = (image_similarities >= 0.40).astype(int)
multimodal_predictions = (final_scores >= 0.20).astype(int)

# Cases where text-only is wrong but multimodal is correct
text_wrong_mm_correct = np.where(
    (text_predictions != image_pair_labels) &
    (multimodal_predictions == image_pair_labels)
)[0]

# Cases where text-only is correct but multimodal is wrong
text_correct_mm_wrong = np.where(
    (text_predictions == image_pair_labels) &
    (multimodal_predictions != image_pair_labels)
)[0]

print("Text wrong → Multimodal correct:", len(text_wrong_mm_correct))
print("Text correct → Multimodal wrong:", len(text_correct_mm_wrong))

Text wrong → Multimodal correct: 32
Text correct → Multimodal wrong: 5


In [25]:
print("MULTIMODAL ERRORS")
print("=" * 80)

for idx in text_correct_mm_wrong:

    i, j = image_pairs[idx]

    image_i = sample_images[i]
    image_j = sample_images[j]

    title_i = sample_meta[
        sample_meta["embedding_index"] == i
    ]["title"].iloc[0]

    title_j = sample_meta[
        sample_meta["embedding_index"] == j
    ]["title"].iloc[0]

    label = image_pair_labels[idx]

    print(f"\nPair {idx}")
    print("Actual:", int(label))
    print("Text similarity:", round(text_similarities[idx], 3))
    print("Image similarity:", round(image_similarities[idx], 3))
    print("Multimodal score:", round(final_scores[idx], 3))
    print("Title 1:", title_i)
    print("Title 2:", title_j)
    print("Image 1:", image_i)
    print("Image 2:", image_j)

MULTIMODAL ERRORS

Pair 349
Actual: 1
Text similarity: 0.185
Image similarity: 0.16
Multimodal score: 0.179
Title 1: [Ready Stock] Ikat Rambut Korea Simple Cute Murah
Title 2: \xe3\x80\x90COD\xe3\x80\x91Korea Versi Dari Tali Rambut Elastis Aksesoris Fashion Ikat Rambut 1PC
Image 1: 2726e2a2bd07a16273d4facec639172c.jpg
Image 2: 45cb4692b7f46bb566e65f2e8c1aef2b.jpg

Pair 360
Actual: 1
Text similarity: 0.166
Image similarity: 0.22
Multimodal score: 0.18
Title 1: \xe3\x80\x90COD\xe3\x80\x91Korea Versi Dari Tali Rambut Elastis Aksesoris Fashion Ikat Rambut 1PC
Title 2: Karet Rambut Wanita Model Korea
Image 1: 45cb4692b7f46bb566e65f2e8c1aef2b.jpg
Image 2: 83bbb79e8e2c6318a0fe91f5ccda99d3.jpg

Pair 361
Actual: 1
Text similarity: 0.131
Image similarity: 0.254
Multimodal score: 0.162
Title 1: \xe3\x80\x90COD\xe3\x80\x91Korea Versi Dari Tali Rambut Elastis Aksesoris Fashion Ikat Rambut 1PC
Title 2: Karet Rambut Wanita Model Korea\xe2\x9d\xa4Yimi\xe2\x9d\xa4
Image 1: 45cb4692b7f46bb566e65f2e8c1ae

In [26]:
print("TEXT ERRORS RESCUED BY MULTIMODAL")
print("=" * 80)

for idx in text_wrong_mm_correct[:10]:

    i, j = image_pairs[idx]

    title_i = sample_meta[
        sample_meta["embedding_index"] == i
    ]["title"].iloc[0]

    title_j = sample_meta[
        sample_meta["embedding_index"] == j
    ]["title"].iloc[0]

    print(f"\nPair {idx}")
    print("Actual:", int(image_pair_labels[idx]))
    print("Text similarity:", round(text_similarities[idx], 3))
    print("Image similarity:", round(image_similarities[idx], 3))
    print("Multimodal score:", round(final_scores[idx], 3))
    print("Title 1:", title_i)
    print("Title 2:", title_j)

TEXT ERRORS RESCUED BY MULTIMODAL

Pair 24
Actual: 1
Text similarity: 0.078
Image similarity: 0.953
Multimodal score: 0.297
Title 1: S-3XL (NO PLASTIK POUCH) PUMA HEAT TECH BISA UTK MANSET ()
Title 2: LIMITED STOCK!!! PUMA Warmwear Multifungsi - Black O Neck PUMA2609-KL

Pair 171
Actual: 1
Text similarity: 0.088
Image similarity: 0.673
Multimodal score: 0.234
Title 1: Viva Milk Cleanser/ Face Tonic 100ml
Title 2: VIVA - AIR MAWAR 100ml \xe2\x9d\xa4 Ewa \xe2\x9d\xa4

Pair 235
Actual: 1
Text similarity: 0.006
Image similarity: 0.864
Multimodal score: 0.221
Title 1: Tas Selempang Quilted Bahan Kulit Quilted Kualitas Tinggi
Title 2: ZF FASHION SLINGBAG RUMBAI TALI RANTAI FASHION WANITA

Pair 267
Actual: 1
Text similarity: 0.088
Image similarity: 0.868
Multimodal score: 0.283
Title 1: Lampu Selang RGB 2835 SMD / Lampu Selang Kotak Warna Warni
Title 2: Lampu Hias Strip LED 10 meter LED Strip Lamp Water Proof Tahan Air Rope Light Kelap Kelip

Pair 273
Actual: 1
Text similarity: 0.094
Image si

In [27]:
print("=" * 75)
print("FINAL MODEL COMPARISON")
print("=" * 75)

print(f"{'Model':<30} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 75)

print(f"{'Image-only (ResNet50)':<30} {'0.9205':>10} {'0.9171':>10} {'0.9245':>10} {'0.9208':>10}")
print(f"{'Text-only (Char TF-IDF)':<30} {'0.9779':>10} {'0.9917':>10} {'0.9638':>10} {'0.9775':>10}")
print(f"{'Multimodal (75/25)':<30} {'0.9863':>10} {'0.9949':>10} {'0.9775':>10} {'0.9862':>10}")
print(f"{'Learned Fusion':<30} {'0.9860':>10} {'0.9785':>10} {'0.9938':>10} {'0.9861':>10}")

FINAL MODEL COMPARISON
Model                            Accuracy  Precision     Recall         F1
---------------------------------------------------------------------------
Image-only (ResNet50)              0.9205     0.9171     0.9245     0.9208
Text-only (Char TF-IDF)            0.9779     0.9917     0.9638     0.9775
Multimodal (75/25)                 0.9863     0.9949     0.9775     0.9862
Learned Fusion                     0.9860     0.9785     0.9938     0.9861
